# AGN kNN-CDF sensitivity to cosmology and feedback

Question: for luminosity-selected AGN (top 10% by bolometric luminosity, above a
`1e6 Msun` base BH-mass floor) at snapshot 50, how does the kNN-CDF clustering
statistic (k = 1, 2, 4) respond to each of the 6 CAMELS-IllustrisTNG LH
parameters -- and at what spatial scale?

Steps:
1. Generate (or load) the AGN kNN-CDF summary for every LH simulation
2. Load parameters, align to the summary's sim_id order, remove the abundance trend
3. Selection-bias diagnostics (this is a luminosity cut -- check it isn't confounding the parameters)
4. Scale-resolved + scalar sensitivity, with bootstrap CI and a permutation null
5. Figures: per-parameter scale response (cosmological vs. astrophysical), summary ranking

**Before running**: point `SIM_PATH` and `PARAMS_FILE` below at your local CAMELS
data (defaults match the layout in `src/config.py`).

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np

from src import config
from src.pipeline import run_suite
from src.params import load_params, align_to_params
from src.abundance import remove_abundance
from src.selection_bias import diagnose_retention_bias, diagnose_nbh_confound, residualize_theta_on_nbh
from src.sensitivity import sensitivity_table, summary_dataframe, infer_layout
from src.plotting import plot_scale_grid, plot_sensitivity_bar

SIM_PATH = config.SIM_PATH
PARAMS_FILE = config.PARAMS_FILE
OUTPUT_DIR = config.OUTPUT_DIR

## 1. Generate (or load) the AGN kNN-CDF summaries

Set `GENERATE = False` once the `.npz` already exists in `OUTPUT_DIR` to skip
the expensive multiprocessing re-run.

In [ ]:
GENERATE = True

pct = int(round(config.TOP_FRACTION * 100))
npz_path = f"{OUTPUT_DIR}/agn_knn_snap{config.SNAP}_M{config.MASS_CUT:.0e}_top{pct}.npz"

if GENERATE:
    result = run_suite(sim_path=SIM_PATH, output_dir=OUTPUT_DIR)
else:
    data = np.load(npz_path, allow_pickle=True)
    result = {
        "sim_ids": data["sim_ids"],
        "summaries": data["summaries"],
        "nbh": data["nbh"],
        "rgrid": data["rgrid"],
        "kvals": data["kvals"],
    }

sim_ids = result["sim_ids"]
summaries = result["summaries"]
nbh = result["nbh"]
rgrid = result["rgrid"]
kvals = result["kvals"]
n_k, n_r = infer_layout(kvals, rgrid)

print(f"{len(sim_ids)} simulations retained, summary shape {summaries.shape}")

## 2. Parameters, alignment, abundance removal

`align_to_params` guarantees `theta.index == sim_ids` row-for-row -- this is
the step that a positional (rather than sim_id-keyed) join got wrong in the
old pipeline, so it's asserted here rather than assumed.

In [ ]:
theta_all = load_params(PARAMS_FILE)
theta = align_to_params(sim_ids, theta_all)

residuals = remove_abundance(summaries, nbh)

assert list(theta.index) == list(sim_ids)
assert residuals.shape[0] == len(theta)

## 3. Selection-bias diagnostics

Two checks, since this is a luminosity (not mass-only) selection:

- **Retention**: did any whole simulations get dropped (too few AGN to reach
  `MIN_AGN`) in a way that correlates with a parameter?
- **nbh confound**: among retained sims, does the *surviving AGN count* trend
  with a parameter? If so, `remove_abundance` may be removing part of that
  parameter's real signal along with the abundance trend -- `residualize_theta_on_nbh`
  is available below to correct for it on any parameter this flags.

In [ ]:
all_ids = theta_all.index.to_numpy()

retention_diag = diagnose_retention_bias(sim_ids, all_ids, theta_all, params=config.ALL_PARAMS)
display(retention_diag)

nbh_diag = diagnose_nbh_confound(nbh, theta, params=config.ALL_PARAMS)
display(nbh_diag)

confounded_params = nbh_diag.loc[nbh_diag["bias_flag"], "parameter"].tolist()
print("Flagged as nbh-confounded:", confounded_params or "none")

# Only apply the correction to parameters actually flagged above.
theta_corrected = residualize_theta_on_nbh(theta, nbh, params=confounded_params) if confounded_params else theta

## 4. Sensitivity: scale-resolved + scalar

Uses `theta_corrected` (== `theta` unless step 3 flagged a confound), so any
correction found above is automatically reflected here.

In [ ]:
table = sensitivity_table(
    residuals, theta_corrected,
    n_k=n_k, rgrid=rgrid, kvals=kvals,
    params=config.ALL_PARAMS,
    n_boot=2000, n_null=2000,
)

summary_df = summary_dataframe(table)
summary_df.sort_values("R_obs", ascending=False)

## 5. Figures

In [ ]:
fig, axes = plot_scale_grid(table, params=config.COSMO_PARAMS)
fig.suptitle("Cosmological parameters", y=1.02)

In [ ]:
fig, axes = plot_scale_grid(table, params=config.ASTRO_PARAMS)
fig.suptitle("Astrophysical (feedback) parameters", y=1.02)

In [ ]:
fig, ax = plot_sensitivity_bar(summary_df, config.COSMO_PARAMS, config.ASTRO_PARAMS)

## Next steps (not built yet)

This notebook answers *whether and where* each parameter imprints on the AGN
kNN-CDF, one parameter at a time. It doesn't yet address **degeneracy**
-- whether two parameters (e.g. `Omega_m` and `A_AGN1`) leave similar-looking
imprints that would be hard to tell apart from the kNN-CDF alone. That's the
planned follow-up, once this single-parameter picture is validated on real
data.